In [5]:
import cv2
import numpy as np

def overlay_heatmap(image, saliency, alpha=0.5):

    image = (image - image.min()) / (image.max() + 1e-8)
    saliency = (saliency - saliency.min()) / (saliency.max() + 1e-8)

    image_uint8 = np.uint8(255 * image)
    heatmap_uint8 = np.uint8(255 * saliency)

    heatmap = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)

    overlay = cv2.addWeighted(
        heatmap,
        alpha,
        np.stack([image_uint8]*3, axis=-1),
        1-alpha,
        0
    )

    return overlay

In [7]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("/home/jovyan/work/MST")
sys.path.append(str(PROJECT_ROOT))

from mst.data.datasets.dataset_3d_odelia import ODELIA_Dataset3D

ds = ODELIA_Dataset3D(
    path_root="/home/jovyan/work/MST/mst/data/datasets/ODELIA_datasets",
    split="test"
)

In [8]:
import json

#Class 0: 02F4A1FB_right
#uid = "02F4A1FB_right"


#data = {"class_0": "02F4A1FB_right", "class_1": "3E6C31E4_left", "class_2": "TRICKS_0067_1_left"}

## Load the JSON file containing the correct UIDs for each class
with open("/home/jovyan/work/MST/scripts/Jupyter_notebook/Confusion Matrix/v3_correct_uids_per_class.json", "r") as f:
    data = json.load(f)
saliency_set = {"GradCAM": "gradcam", "Raw Attention": "last_layer", "Slice Weighted Rollout": "slice_weighted_rollout"}



In [9]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
import gc


def save_saliency_visualizations(
    data,
    saliency_set,
    dataset,
    overlay_heatmap,
    save_root,
    num_slices=32,
):
    """
    Save saliency visualizations (overlay + comparison) for all samples.

    Args:
        data (dict): {class_name: uid}
        saliency_set (dict): {method_name: saliency_folder}
        dataset (Dataset): dataset containing samples
        overlay_heatmap (function): function(img, sal) -> RGB overlay
        save_root (str): directory to save images
        num_slices (int): number of slices (default 32)
    """

    os.makedirs(save_root, exist_ok=True)

    for class_name, uids in data.items():
        class_save_root = os.path.join(save_root, class_name)
        os.makedirs(class_save_root, exist_ok=True)

        for uid in uids:
            for method_name, saliency_map in saliency_set.items():

                print(f"Processing: {uid} | {class_name} | {method_name}")

                # -----------------------------
                # FIND SAMPLE
                # -----------------------------
                sample = None
                for i in range(len(dataset)):
                    s = dataset[i]
                    if s["uid"] == uid:
                        sample = s
                        break

                if sample is None:
                    print(f"[WARNING] UID not found: {uid}")
                    continue

                volume = sample["source"][0].numpy()  # [32,224,224]
                label = sample["target"]

                # -----------------------------
                # LOAD SALIENCY
                # -----------------------------
                saliency_path = (
                    f"/home/jovyan/work/MST/results/DINOv3ViTB/saliency_results/"
                    f"{saliency_map}/class_{label}/pt/{uid}_importance.pt"
                )

                if not os.path.exists(saliency_path):
                    print(f"[WARNING] Missing saliency: {saliency_path}")
                    continue

                saliency = torch.load(saliency_path, map_location="cpu").numpy()

                # =========================================================
                # 1) OVERLAY GRID
                # =========================================================
                overlays = []
                for i in range(num_slices):
                    overlays.append(
                        overlay_heatmap(volume[i], saliency[i])
                    )

                fig, axes = plt.subplots(4, 8, figsize=(20, 10))
                for i, ax in enumerate(axes.flat):
                    ax.imshow(overlays[i])
                    ax.set_title(f"S{i+1}", fontsize=8)
                    ax.axis("off")

                fig.suptitle(f"{uid} | {class_name} | {method_name}", fontsize=14)

                overlay_path = os.path.join(
                    class_save_root,
                    f"{uid}_{class_name}_{method_name}_overlay.png"
                )

                plt.tight_layout()
                plt.savefig(overlay_path, dpi=150)
                plt.close(fig)

                # =========================================================
                # 2) COMPARISON (RAW + OVERLAY)
                # =========================================================
                comparisons = []
                for i in range(num_slices):

                    img = volume[i]
                    sal = saliency[i]

                    overlay = overlay_heatmap(img, sal)

                    img_rgb = np.stack([img]*3, axis=-1)
                    img_rgb = (img_rgb - img_rgb.min()) / (img_rgb.max() + 1e-8)
                    img_rgb = np.uint8(255 * img_rgb)

                    combined = np.concatenate([img_rgb, overlay], axis=1)
                    comparisons.append(combined)

                fig, axes = plt.subplots(8, 4, figsize=(10, 12))
                for i, ax in enumerate(axes.flat):
                    ax.imshow(comparisons[i])
                    ax.set_title(f"S{i+1}", fontsize=8)
                    ax.axis("off")

                fig.suptitle(f"{uid} | {class_name} | {method_name}", fontsize=14)

                comparison_path = os.path.join(
                    class_save_root,
                    f"{uid}_{class_name}_{method_name}_comparison.png"
                )

                plt.tight_layout()
                plt.savefig(comparison_path, dpi=150)
                plt.close(fig)

                # -----------------------------
                # MEMORY CLEANUP
                # -----------------------------
                del overlays, comparisons, saliency, volume
                torch.cuda.empty_cache()
                gc.collect()

In [ ]:
save_saliency_visualizations(
    data=data,
    saliency_set=saliency_set,
    dataset=ds,
    overlay_heatmap=overlay_heatmap,
    save_root="/home/jovyan/work/MST/scripts/Jupyter_notebook/Overlay Image/png"
)

Processing: ODELIA_BRAID1_0246_1_left | 0 | GradCAM


/tmp/ipykernel_3964/2754434347.py:68: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  saliency = torch.load(saliency_path, map_location="cpu").numpy()


Processing: ODELIA_BRAID1_0246_1_left | 0 | Raw Attention
Processing: ODELIA_BRAID1_0246_1_left | 0 | Slice Weighted Rollout
Processing: ODELIA_TRICKS_0067_1_right | 0 | GradCAM
Processing: ODELIA_TRICKS_0067_1_right | 0 | Raw Attention
Processing: ODELIA_TRICKS_0067_1_right | 0 | Slice Weighted Rollout
Processing: ODELIA_BRAID1_0187_1_left | 0 | GradCAM
Processing: ODELIA_BRAID1_0187_1_left | 0 | Raw Attention
Processing: ODELIA_BRAID1_0187_1_left | 0 | Slice Weighted Rollout
Processing: ODELIA_BRAID1_0777_1_left | 0 | GradCAM
Processing: ODELIA_BRAID1_0777_1_left | 0 | Raw Attention
Processing: ODELIA_BRAID1_0777_1_left | 0 | Slice Weighted Rollout
Processing: ODELIA_TRICKS_0101_1_left | 0 | GradCAM
Processing: ODELIA_TRICKS_0101_1_left | 0 | Raw Attention
Processing: ODELIA_TRICKS_0101_1_left | 0 | Slice Weighted Rollout
Processing: ODELIA_BRAID1_0187_1_right | 0 | GradCAM
Processing: ODELIA_BRAID1_0187_1_right | 0 | Raw Attention
Processing: ODELIA_BRAID1_0187_1_right | 0 | Slice W